# Генерация новых жалоб (ЖКТ + КОЖА): baseline vs трансформер

В отличие от перефразирования, тут LLM придумывает **новые** жалобы по каждому диагнозу
слабых классов — добавляем новую информацию, а не переписываем старую.
Сравниваем на dev с результатами без аугментации (baseline 0.841, трансформер 0.854),
затем — один раз на test против 0.865.

**Перед запуском:** Runtime → GPU.

In [ ]:
%cd /content
!rm -rf anamnesis && git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
!pip install -q transformers accelerate peft
!pip uninstall -q -y torchao

In [ ]:
import getpass, os
os.environ["MISTRAL_API_KEY"] = getpass.getpass("MISTRAL_API_KEY: ")

## 1. Генерация новых жалоб

Добиваем ЖКТ и КОЖА до 500, генерируя новые случаи по каждому диагнозу (создаёт `data/train_gen_v1.jsonl`).

In [ ]:
from src.generate import main as generate
generate(target_per_class=500, groups=["ЖКТ/БРЮШНЫЕ", "КОЖА"])

## 2. Baseline (tf-idf + LogReg) на сгенерированных данных

Сравнение без генерации: dev macro-F1 0.841.

In [ ]:
from src.baseline import build_model
from src.data import load_xy
from src.evaluate import scores, print_report
from src.mapping import GROUPS

x_tr, y_tr = load_xy("train_gen")
x_dv, y_dv = load_xy("dev")
clf = build_model().fit(x_tr, y_tr)
pred = clf.predict(x_dv)
print("baseline+gen on dev:", scores(y_dv, pred))
print_report(y_dv, pred, labels=list(GROUPS))

## 3. Трансформер на сгенерированных данных (dev)

Сравнение без генерации: dev macro-F1 0.854. Ранняя остановка по dev.

In [ ]:
from src.transformer_ft_aug import train
train(eval_split="dev", train_split="train_gen", name="rubioroberta_ft_gen")

## 4. Финальный замер на test (только если на dev стало лучше)

Грузит обученную на dev модель и считает test один раз — без переобучения и без утечки.

In [ ]:
from src.transformer_ft_aug import evaluate_saved
evaluate_saved(eval_split="test", name="rubioroberta_ft_gen")

## Метрики + сохранение в git

In [ ]:
import json
print(json.dumps(json.load(open("reports/metrics.json")), ensure_ascii=False, indent=2))

In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git config user.email "karinkakarik@mail.ru"
!git config user.name "Git_Karina"
!git add data/train_gen_v1.jsonl reports/
!git commit -q -m "generation results"
!git pull -q --rebase https://github.com/vinograd131/anamnesis.git main
!git push -q https://{token}@github.com/vinograd131/anamnesis.git HEAD:main